# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
* Model: BERT
* Evaluation approach: evaluate method
* Fine-tuning dataset: cornell-movie-review-data/rotten_tomatoes

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
from datasets import load_dataset
#Loading dataset and splitting it into train and test
splits = ["train", "test"]
dataset = {split: dataset for split, dataset in zip(splits, load_dataset("cornell-movie-review-data/rotten_tomatoes", split=splits))}

for split in splits:
    dataset[split] = dataset[split].shuffle(seed=42).select(range(50))

dataset

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

{'train': Dataset({
     features: ['text', 'label'],
     num_rows: 50
 }),
 'test': Dataset({
     features: ['text', 'label'],
     num_rows: 50
 })}

In [2]:
from transformers import AutoTokenizer
#Loading tokenizer and applying it to both test and train datasets
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = {}
for split in splits:
    tokenized_datasets[split] = dataset[split].map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoModelForSequenceClassification
#Loading the pretrained distilbert model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"}, #Labels represent positive and negative movie reviews
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.bias', 'classifier.weight', 'classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
#metrics used to calculate the accuracy of the models
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

#training arguments for the pretrained model
training_args = TrainingArguments(output_dir="test_trainer", learning_rate=2e-4, evaluation_strategy="epoch", num_train_epochs=1, per_device_train_batch_size=2,)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorWithPadding(tokenizer = tokenizer),
    compute_metrics=compute_metrics,
)

/opt/conda/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


/opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32


In [5]:
trainer.evaluate()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'eval_loss': 0.7011018991470337,
 'eval_accuracy': 0.48,
 'eval_runtime': 86.5942,
 'eval_samples_per_second': 0.577,
 'eval_steps_per_second': 0.081}

## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [6]:
from peft import LoraConfig, get_peft_model

#loading lora config with task_type, inference_mode, r, lora_alpha, lora_dropout, and target_modules parameters for 
#this chosen dataset
config = LoraConfig(task_type="SEQ_CLS", inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.1, target_modules = ['q_lin', 'v_lin'])

In [7]:
#loading the peft model
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 1,331,716 || all params: 67,694,596 || trainable%: 1.967241225577297


In [8]:
import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
#training the lora model 
lora_training_args = TrainingArguments(output_dir="lora_trainer",
                                        evaluation_strategy='steps',
                                        learning_rate=2e-4,
                                        num_train_epochs=1,
                                        per_device_train_batch_size=2, )

lora_trainer = Trainer(
    model = lora_model,
    args = lora_training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
    tokenizer = tokenizer,
    data_collator = DataCollatorWithPadding(tokenizer = tokenizer),
    compute_metrics = compute_metrics,
)

lora_trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=25, training_loss=0.7300980377197266, metrics={'train_runtime': 201.436, 'train_samples_per_second': 0.248, 'train_steps_per_second': 0.124, 'total_flos': 6736970342400.0, 'train_loss': 0.7300980377197266, 'epoch': 1.0})

In [9]:
lora_model.print_trainable_parameters()

trainable params: 1,331,716 || all params: 67,694,596 || trainable%: 1.967241225577297


In [10]:
lora_model.save_pretrained("distilbert-base-peft")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [11]:
from peft import AutoPeftModelForSequenceClassification
#loading the peft model
peft_model = AutoPeftModelForSequenceClassification.from_pretrained("distilbert-base-peft")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.bias', 'classifier.weight', 'classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
peft_training_args = TrainingArguments(output_dir="peft_trainer",
                                        evaluation_strategy='steps',
                                        learning_rate=2e-4,
                                        num_train_epochs=10,
                                        per_device_train_batch_size=2, )

#training the peft model for evaluation and comparison
peft_trainer = Trainer(
    model = peft_model,
    args = peft_training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
    tokenizer = tokenizer,
    data_collator = DataCollatorWithPadding(tokenizer = tokenizer),
    compute_metrics = compute_metrics,
)

peft_trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=250, training_loss=0.5280449829101562, metrics={'train_runtime': 954.4858, 'train_samples_per_second': 0.524, 'train_steps_per_second': 0.262, 'total_flos': 67369703424000.0, 'train_loss': 0.5280449829101562, 'epoch': 10.0})

In [13]:
#peft model shows a substantial increase in accuracy compared to the pretrained model
peft_trainer.evaluate()

{'eval_loss': 0.5868499875068665,
 'eval_accuracy': 0.7,
 'eval_runtime': 67.791,
 'eval_samples_per_second': 0.738,
 'eval_steps_per_second': 0.103,
 'epoch': 10.0}

In [14]:
trainer.evaluate()

{'eval_loss': 0.7170923352241516,
 'eval_accuracy': 0.48,
 'eval_runtime': 68.6996,
 'eval_samples_per_second': 0.728,
 'eval_steps_per_second': 0.102}